In [1]:
%pwd

'a:\\projects\\text-summarizer\\research'

In [2]:
import os
os.chdir('../')

In [3]:
%pwd

'a:\\projects\\text-summarizer'

# Entity

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: str

# Configuration manager

In [5]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories
from pathlib import Path

In [6]:
class ConfigurationManager:
    def __init__(self, 
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories(path_to_directories=[Path(self.config.artifacts_root)])
    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        root_dir = Path(config.root_dir)
        data_path = Path(config.data_path)

        create_directories(path_to_directories=[root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=root_dir,
            data_path=data_path,
            tokenizer_name=str(config.tokenizer_name)
        )

        return data_transformation_config

# Components

In [12]:
import os
from textSummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset

In [13]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)

    def convert_examples_to_features(self, example_batch):
        input_encodings = self.tokenizer(example_batch["dialogue"], 
                                         max_length = 1024, truncation=True)

        target_encodings = self.tokenizer(text_target=example_batch["summary"], 
                                          max_length = 128, truncation=True)

        return {
            "input_ids": input_encodings["input_ids"],
            "attention_mask": input_encodings["attention_mask"],
            "labels": target_encodings["input_ids"]
        }
    
    def convert(self):
        try:
            dataset_samsum = load_dataset('json', data_files={
                'train': os.path.join(self.config.data_path, 'train.json'),
                'test': os.path.join(self.config.data_path, 'test.json'),
                'validation': os.path.join(self.config.data_path, 'val.json')
            })
            
            dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched=True)
            
            output_path = os.path.join(self.config.root_dir, "samsum_dataset")
            dataset_samsum_pt.save_to_disk(output_path) 
            
        except Exception as e:
            raise e

# Pipeline

In [14]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e

[2026-06-19 19:02:31,854: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-19 19:02:31,862: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-19 19:02:31,865: INFO: common: Created directory at: artifacts]
[2026-06-19 19:02:31,868: INFO: common: Created directory at: artifacts\data_transformation]
[2026-06-19 19:02:32,470: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-06-19 19:02:32,565: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-06-19 19:02:32,911: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-06-19 19:02:33,373: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-c

Generating train split: 14732 examples [00:00, 28511.85 examples/s]
Generating test split: 819 examples [00:00, 22677.75 examples/s]
Generating validation split: 818 examples [00:00, 24928.55 examples/s]
Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 42141.90 examples/s]
